\title{}
\author{}
\date{}
\makeatletter
\renewcommand{\maketitle}{}
\makeatother

\thispagestyle{empty}

\begin{center}
\vspace*{4cm}

{\LARGE Asset Allocation \& Investment Strategies \\[0.5cm]}

Academic year: 2025--2026\\[1.5cm]

Group 8\\[0.3cm]
Sacha Mimoun\\
Isabelle Chuah\\
Victor Lotigie\\
Evelyn Wang\\
Bolun Tian\\[1.5cm]

\textit{Imperial College Business School}

\end{center}

\newpage

\setcounter{secnumdepth}{0}

\thispagestyle{empty}
\clearpage

\tableofcontents

\newpage

In [2]:
import pandas as pd
import datetime
import os
import statsmodels.api as sm

In [3]:
# Import raw data

path = "raw_data/"
filename_factors = 'F-F_Research_Data_Factors.CSV'
filename_portfolios = '25_Portfolios_5x5.CSV'
filepath = os.path.join(path, filename_factors)
filepath_portfolios = os.path.join(path, filename_portfolios)

In [4]:
# Read F-F factors file - extract only monthly data
df_temp = pd.read_csv(filepath, skiprows=3, header=None)
separator_idx = df_temp[df_temp[0].astype(str).str.contains('Annual', case=False, na=False)].index[0]



In [5]:
# Read only monthly data (stop before annual section)
df_ff_monthly = pd.read_csv(filepath, skiprows=3, nrows=separator_idx-1)
# Clean monthly factors data
df_ff_monthly['date_str'] = df_ff_monthly.iloc[:, 0].astype(str).str.strip()
df_ff_monthly = df_ff_monthly[df_ff_monthly['date_str'].str.match(r'^\d{6}$')]
df_ff_monthly.index = pd.to_datetime(df_ff_monthly['date_str'], format='%Y%m').dt.strftime('%Y-%m-%d')
df_ff_monthly = df_ff_monthly.iloc[:, 1:-1] / 100 # because it was in percentage before
print("Fama-French Monthly Factors:")
print(df_ff_monthly)
print(f"Shape: {df_ff_monthly.shape}")

Fama-French Monthly Factors:
            Mkt-RF     SMB     HML      RF
date_str                                  
1926-07-01  0.0296 -0.0256 -0.0243  0.0022
1926-08-01  0.0264 -0.0117  0.0382  0.0025
1926-09-01  0.0036 -0.0140  0.0013  0.0023
1926-10-01 -0.0324 -0.0009  0.0070  0.0032
1926-11-01  0.0253 -0.0010 -0.0051  0.0031
...            ...     ...     ...     ...
2023-08-01 -0.0239 -0.0316 -0.0106  0.0045
2023-09-01 -0.0524 -0.0251  0.0152  0.0043
2023-10-01 -0.0319 -0.0387  0.0019  0.0047
2023-11-01  0.0884 -0.0002  0.0164  0.0044
2023-12-01  0.0485  0.0635  0.0494  0.0043

[1170 rows x 4 columns]
Shape: (1170, 4)


Let's do the same for the 25 portfolios

In [6]:
# Read 25 Portfolios file - extract only monthly datasets
with open(filepath_portfolios, 'r') as f:
    lines = f.readlines()

# Find indices for monthly datasets
vw_idx = next(i for i, line in enumerate(lines) if 'Average Value Weighted Returns -- Monthly' in line)
ew_idx = next(i for i, line in enumerate(lines) if 'Average Equal Weighted Returns -- Monthly' in line)

print(f"VW Monthly: line {vw_idx}, EW Monthly: line {ew_idx}")

VW Monthly: line 14, EW Monthly: line 1188


In [7]:
# Extract monthly datasets
df_port_vw = pd.read_csv(filepath_portfolios, skiprows=vw_idx+1, nrows=ew_idx-vw_idx-3)
df_port_vw['Date'] = df_port_vw.iloc[:, 0].astype(str).str.strip()
df_port_vw = df_port_vw[df_port_vw['Date'].str.match(r'^\d{6}$', na=False)]
df_port_vw.index = pd.to_datetime(df_port_vw['Date'], format='%Y%m').dt.strftime('%Y-%m-%d')
df_port_vw = df_port_vw.iloc[:, 1:-1] / 100

df_port_ew = pd.read_csv(filepath_portfolios, skiprows=ew_idx+1, nrows=ew_idx-vw_idx-3)
df_port_ew['Date'] = df_port_ew.iloc[:, 0].astype(str).str.strip()
df_port_ew = df_port_ew[df_port_ew['Date'].str.match(r'^\d{6}$', na=False)]
df_port_ew.index = pd.to_datetime(df_port_ew['Date'], format='%Y%m').dt.strftime('%Y-%m-%d')
df_port_ew = df_port_ew.iloc[:, 1:-1] / 100 # it was again in percentage

print(f"\nVW Monthly: {df_port_vw.shape}")
print(df_port_vw)

print(f"EW Monthly: {df_port_ew.shape}")

#print(df_port_ew)


VW Monthly: (1170, 25)
            SMALL LoBM   ME1 BM2   ME1 BM3   ME1 BM4  SMALL HiBM   ME2 BM1  \
Date                                                                         
1926-07-01    0.058248 -0.017006  0.004875 -0.014580    0.020534  0.012077   
1926-08-01   -0.020206 -0.080282  0.013796  0.014606    0.083968  0.023618   
1926-09-01   -0.048291 -0.026154 -0.043417 -0.032729    0.008649 -0.026540   
1926-10-01   -0.093729 -0.035519 -0.034948  0.034413   -0.025476 -0.028069   
1926-11-01    0.055888  0.041877  0.024623 -0.044494    0.005362  0.031033   
...                ...       ...       ...       ...         ...       ...   
2023-08-01   -0.122376 -0.076627 -0.105103 -0.056405   -0.073431 -0.069345   
2023-09-01   -0.082275 -0.076201 -0.065856 -0.058185   -0.060216 -0.088904   
2023-10-01   -0.102240 -0.089443 -0.079196 -0.063774   -0.076783 -0.102040   
2023-11-01    0.057861  0.080737  0.107319  0.085040    0.070945  0.109883   
2023-12-01    0.153216  0.160383  0.1486

In [8]:
# Compute excess returns for the 25 portfolios
df_port_vw_excess = df_port_vw.sub(df_ff_monthly['RF'], axis=0)
df_port_ew_excess = df_port_ew.sub(df_ff_monthly['RF'], axis=0)

In [18]:
def slice_monthly(df, start, end):
    df.index = pd.to_datetime(df.index)
    start = pd.to_datetime(start)
    end = pd.to_datetime(end)
    return df.loc[(df.index >= start) & (df.index <= end)].copy()

start, end = "1964-01-01", "1993-01-01"

factors_6493 = slice_monthly(df_ff_monthly, start, end)
vw_port_excess_6493 = slice_monthly(df_port_vw_excess, start, end)
# ew_port_excess_6493 = slice_monthly(df_port_vw_excess, start, end)

common_idx = factors_6493.index.intersection(vw_port_excess_6493.index)
factors_6493 = factors_6493.loc[common_idx]
ports_6493 = vw_port_excess_6493.loc[common_idx]

print("Sample range:", common_idx.min(), "→", common_idx.max())
print("T =", len(common_idx))
print("Aligned:", factors_6493.index.equals(ports_6493.index))

Sample range: 1964-01-01 00:00:00 → 1993-01-01 00:00:00
T = 349
Aligned: True


In [21]:
print(ports_6493)

            SMALL LoBM   ME1 BM2   ME1 BM3   ME1 BM4  SMALL HiBM   ME2 BM1  \
1964-01-01    0.036847  0.026078  0.039205  0.038486    0.045526 -0.023215   
1964-02-01    0.026988  0.025947  0.020073  0.000296    0.041942  0.013574   
1964-03-01    0.001930  0.011981  0.018931  0.032385    0.025121  0.020333   
1964-04-01   -0.018527  0.022981 -0.007426 -0.011988   -0.012124 -0.011990   
1964-05-01   -0.015118 -0.015745  0.002791  0.000878    0.011697  0.028853   
...                ...       ...       ...       ...         ...       ...   
1992-09-01    0.002120  0.012340  0.015981  0.019947    0.008409  0.012908   
1992-10-01    0.023034  0.017113  0.023651  0.017618    0.013572  0.045488   
1992-11-01    0.104459  0.106869  0.079416  0.080027    0.076541  0.090628   
1992-12-01    0.012935  0.028679  0.047902  0.041556    0.047862  0.012840   
1993-01-01    0.023874  0.052866  0.040558  0.043602    0.077916 -0.010212   

             ME2 BM2   ME2 BM3   ME2 BM4   ME2 BM5  ...   ME4 B

In [29]:
# Prepare results storage
results = []

mkt_rf = factors_6493['Mkt-RF']

for col in vw_port_excess_6493.columns:
    y = vw_port_excess_6493[col]
    X = sm.add_constant(mkt_rf)
    model = sm.OLS(y, X).fit()
    
    results.append({
        'portfolio': col,
        "mean_excess": y.mean(),
        'alpha': model.params['const'],
        'beta': model.params['Mkt-RF'],
        'alpha_tstat': model.tvalues['const'],
        'beta_tstat': model.tvalues['Mkt-RF'],
        "adj_r2": model.rsquared_adj
    })

df_results = pd.DataFrame(results)

print("\n Regression Results :")
print(df_results.to_string(index=False))



 Regression Results :
 portfolio  mean_excess     alpha     beta  alpha_tstat  beta_tstat   adj_r2
SMALL LoBM     0.003090 -0.002637 1.425884    -1.126317   27.798851 0.689224
   ME1 BM2     0.007425  0.002403 1.250530     1.186556   28.182754 0.695076
   ME1 BM3     0.007840  0.003186 1.158853     1.726040   28.657677 0.702122
   ME1 BM4     0.009437  0.005112 1.077028     2.793956   26.867342 0.674418
SMALL HiBM     0.010844  0.006414 1.103020     3.081514   24.185137 0.626578
   ME2 BM1     0.004047 -0.001689 1.428427    -0.962518   37.147215 0.798485
   ME2 BM2     0.006582  0.001625 1.234521     1.079549   37.442397 0.801022
   ME2 BM3     0.008828  0.004346 1.116015     3.037569   35.597026 0.784406
   ME2 BM4     0.009565  0.005416 1.033107     3.991288   34.744860 0.776091
   ME2 BM5     0.010755  0.006243 1.123314     3.663913   30.086892 0.722094
   ME3 BM1     0.004491 -0.000945 1.353789    -0.697838   45.604053 0.856597
   ME3 BM2     0.006975  0.002305 1.162876     1.9812

In [ ]:
size_order = ["SMALL", "ME2", "ME3", "ME4", "BIG"]
size_names = ["Small", "2", "3", "4", "Big"]

bm_order = ["LoBM", "BM2", "BM3", "BM4", "HiBM"]
bm_names = ["Low", "2", "3", "4", "High"]

def parse_ff_portfolio_label(label: str):
    """
    Returns (size_bucket, bm_bucket) from labels like:
    - 'SMALL LoBM' -> ('SMALL', 'LoBM')
    - 'ME3 BM4' -> ('ME3', 'BM4')
    - 'BIG HiBM' -> ('BIG', 'HiBM')

    Note: ME rows use BM1/BM5 sometimes, which correspond to LoBM/HiBM.
    """
    parts = label.split()
    if len(parts) != 2:
        return None

    size, bm = parts[0], parts[1]

    if size not in {"SMALL", "ME1", "ME2", "ME3", "ME4", "ME5", "BIG"}:
        return None

    # Normalize size naming: treat ME1 as SMALL row, ME5 as BIG row (FF conventions)
    if size == "ME1":
        size = "SMALL"
    if size == "ME5":
        size = "BIG"

    # Normalize BM endpoints
    if bm == "BM1":
        bm = "LoBM"
    if bm == "BM5":
        bm = "HiBM"

    return size, bm

def to_5x5(res_df: pd.DataFrame, col: str) -> pd.DataFrame:
    tbl = pd.DataFrame(np.nan, index=size_names, columns=bm_names)

    for port in res_df.index:
        parsed = parse_ff_portfolio_label(port)
        if parsed is None:
            continue
        size, bm = parsed

        if (size in size_order) and (bm in bm_order):
            i = size_order.index(size)
            j = bm_order.index(bm)
            tbl.iloc[i, j] = res_df.loc[port, col]

    return tbl

tables_6493 = {
    "Mean excess return": to_5x5(capm_6493, "mean_excess"),
    "Beta (rmrf)": to_5x5(capm_6493, "beta"),
    "t(Beta)": to_5x5(capm_6493, "beta_t"),
    "Alpha": to_5x5(capm_6493, "alpha"),
    "t(Alpha)": to_5x5(capm_6493, "alpha_t"),
    "Adj R^2": to_5x5(capm_6493, "adj_r2"),
}

tables_6493["Beta (rmrf)"]

## Part VII: Connecting the portfolios within each b/m category

## Part VIII: Connecting the portfolios within each size category